# Jersey Pipeline (Colab + Resume + Drive Persistence)

Use this notebook for repeatable runs on ephemeral Colab runtimes:

- Keep **repo / dataset zip / weights / outputs** on Drive.
- Run in `/content` (fast local disk) every session.
- Use `--resume` (default) so completed pipeline stages are skipped.
- Use `--force` only when you need a full recompute.

**Changing runtime (CPU ↔ GPU) or reconnecting starts a new VM** — `/content` is wiped. **Re-run all cells from the top** (Drive mount → sync repo → dataset → weights → install) before `main.py`.


In [ ]:
# ===== CONFIG =====
DRIVE_PROJECT = 'jersey-number-pipeline'   # folder under MyDrive
REPO_NAME = 'jersey-number-pipeline'
REPO_URL = 'https://github.com/superbolt08/jersey-number-pipeline.git'

# Dataset zip on Drive
DATASET_ZIP_NAME = 'jersey-2023.zip'

# Weights folder layout on Drive:
# MyDrive/<DRIVE_PROJECT>/weights/models/*
# MyDrive/<DRIVE_PROJECT>/weights/reid/*
# MyDrive/<DRIVE_PROJECT>/weights/pose/*
WEIGHTS_DIR_NAME = 'weights'

# Pipeline run behavior
PART = 'test'               # test / val / challenge
RESUME = True               # True -> --resume
FORCE = False               # True -> --force (overrides resume)

# Keep this False for full tracklet run (recommended for final evaluation)
LIMIT_TRACKLETS = False

# If True, cache extracted dataset folder back to Drive once (large copy, but future runs are faster)
CACHE_EXTRACTED_DATASET_TO_DRIVE = False


In [ ]:
from google.colab import drive
import os, shutil, subprocess, textwrap, pathlib, sys

drive.mount('/content/drive')

def run(cmd, cwd=None):
    """Run shell command; capture merged stdout+stderr so failures show the real traceback."""
    print(f'\n[RUN] {cmd}')
    p = subprocess.run(
        cmd, shell=True, cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if p.stdout:
        print(p.stdout, end='')
    if p.returncode != 0:
        raise RuntimeError(f'Command failed ({p.returncode}): {cmd}')

# Google Drive over FUSE: rsync often exits 23 ("partial transfer") on unreadable shortcuts or temp files.
# Drop --delete here: we rm -rf LOCAL_REPO_DIR first; --delete adds Drive metadata churn.
_RSYNC_FROM_DRIVE = (
    'rsync -rltD --no-perms --no-owner --no-group --modify-window=2 '
    '--copy-links --partial '
    '--exclude=".tmp.drivedownload" --exclude=".Trash*" '
)

def run_rsync_from_drive(src_dir, dst_dir):
    cmd = f'{_RSYNC_FROM_DRIVE} "{src_dir}/" "{dst_dir}/"'
    print(f'\n[RUN] {cmd}')
    p = subprocess.run(cmd, shell=True)
    if p.returncode == 0:
        return
    if p.returncode == 23:
        print('WARN: rsync exit 23 (partial transfer). Common with Drive; if clone looks OK, continue.')
        print('Tip: remove non-file Google shortcuts from the repo folder on Drive, or retry after remount.')
        return
    raise RuntimeError(f"rsync failed ({p.returncode}): {cmd}")

DRIVE_ROOT = '/content/drive/MyDrive'
DRIVE_PROJECT_DIR = os.path.join(DRIVE_ROOT, DRIVE_PROJECT)
DRIVE_REPO_DIR = os.path.join(DRIVE_PROJECT_DIR, REPO_NAME)
DRIVE_DATASET_ZIP = os.path.join(DRIVE_PROJECT_DIR, DATASET_ZIP_NAME)
DRIVE_WEIGHTS_DIR = os.path.join(DRIVE_PROJECT_DIR, WEIGHTS_DIR_NAME)
DRIVE_DATASET_EXTRACTED = os.path.join(DRIVE_PROJECT_DIR, 'data', 'SoccerNet', 'jersey-2023')

LOCAL_REPO_DIR = os.path.join('/content', REPO_NAME)
LOCAL_DATA_DIR = os.path.join(LOCAL_REPO_DIR, 'data', 'SoccerNet')
LOCAL_DATASET_ROOT = os.path.join(LOCAL_DATA_DIR, 'jersey-2023')

os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print('Drive project dir:', DRIVE_PROJECT_DIR)
print('Local repo dir:', LOCAL_REPO_DIR)


In [ ]:
# 1) Ensure repo exists on Drive (persistent), then sync to fast local disk (/content)
if not os.path.isdir(os.path.join(DRIVE_REPO_DIR, '.git')):
    run(f'git clone {REPO_URL} "{DRIVE_REPO_DIR}"')
else:
    run('git pull', cwd=DRIVE_REPO_DIR)

if os.path.isdir(LOCAL_REPO_DIR):
    run(f'rm -rf "{LOCAL_REPO_DIR}"')
run_rsync_from_drive(DRIVE_REPO_DIR, LOCAL_REPO_DIR)

# Clone required sub-repos if missing (into local working copy)
os.makedirs(os.path.join(LOCAL_REPO_DIR, 'reid'), exist_ok=True)
os.makedirs(os.path.join(LOCAL_REPO_DIR, 'pose'), exist_ok=True)
os.makedirs(os.path.join(LOCAL_REPO_DIR, 'str'), exist_ok=True)

# SAM: empty sam2/ from Drive still counts as "exists" — require sam.py or re-clone
_sam2 = os.path.join(LOCAL_REPO_DIR, 'sam2')
if not os.path.isfile(os.path.join(_sam2, 'sam.py')):
    if os.path.isdir(_sam2):
        run(f'rm -rf "{_sam2}"')
    run('git clone --recurse-submodules https://github.com/davda54/sam.git sam2', cwd=LOCAL_REPO_DIR)
if not os.path.isdir(os.path.join(LOCAL_REPO_DIR, 'reid', 'centroids-reid')):
    run('git clone --recurse-submodules https://github.com/mikwieczorek/centroids-reid.git reid/centroids-reid', cwd=LOCAL_REPO_DIR)
if not os.path.isdir(os.path.join(LOCAL_REPO_DIR, 'pose', 'ViTPose')):
    run('git clone --recurse-submodules https://github.com/ViTAE-Transformer/ViTPose.git pose/ViTPose', cwd=LOCAL_REPO_DIR)
if not os.path.isdir(os.path.join(LOCAL_REPO_DIR, 'str', 'parseq')):
    run('git clone --recurse-submodules https://github.com/baudm/parseq.git str/parseq', cwd=LOCAL_REPO_DIR)

print('Repo + sub-repos ready.')

In [ ]:
# 2) Dataset staging (resume-friendly)
# Priority:
#   A) local extracted already present -> skip
#   B) extracted copy exists on Drive -> rsync down
#   C) dataset zip exists on Drive -> copy zip local + unzip local

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

def dataset_ready(root):
    return os.path.isdir(os.path.join(root, 'test', 'images'))

if dataset_ready(LOCAL_DATASET_ROOT):
    print('Dataset already present locally, skipping extraction.')
elif dataset_ready(DRIVE_DATASET_EXTRACTED):
    print('Syncing extracted dataset from Drive -> local /content ...')
    run(f'rsync -a "{DRIVE_DATASET_EXTRACTED}/" "{LOCAL_DATASET_ROOT}/"')
elif os.path.isfile(DRIVE_DATASET_ZIP):
    local_zip = os.path.join('/content', DATASET_ZIP_NAME)
    run(f'cp "{DRIVE_DATASET_ZIP}" "{local_zip}"')
    run(f'unzip -o -q "{local_zip}" -d "{LOCAL_DATA_DIR}"')
    # If zip expanded to train/test directly, move under jersey-2023
    train_dir = os.path.join(LOCAL_DATA_DIR, 'train')
    test_dir = os.path.join(LOCAL_DATA_DIR, 'test')
    if os.path.isdir(train_dir) and os.path.isdir(test_dir) and not os.path.isdir(LOCAL_DATASET_ROOT):
      os.makedirs(LOCAL_DATASET_ROOT, exist_ok=True)
      run(f'mv "{train_dir}" "{LOCAL_DATASET_ROOT}/"')
      run(f'mv "{test_dir}" "{LOCAL_DATASET_ROOT}/"')
    if CACHE_EXTRACTED_DATASET_TO_DRIVE:
      os.makedirs(os.path.dirname(DRIVE_DATASET_EXTRACTED), exist_ok=True)
      run(f'rsync -a "{LOCAL_DATASET_ROOT}/" "{DRIVE_DATASET_EXTRACTED}/"')
else:
    raise FileNotFoundError(
        f'Dataset not found. Expected one of:\n'
        f' - extracted: {DRIVE_DATASET_EXTRACTED}\n'
        f' - zip:       {DRIVE_DATASET_ZIP}'
    )

assert dataset_ready(LOCAL_DATASET_ROOT), f'Dataset layout invalid at {LOCAL_DATASET_ROOT}'
print('Dataset ready at', LOCAL_DATASET_ROOT)


In [ ]:
# 3) Copy weights from Drive to local repo paths
models_dst = os.path.join(LOCAL_REPO_DIR, 'models')
reid_dst = os.path.join(LOCAL_REPO_DIR, 'reid', 'centroids-reid', 'models')
pose_dst = os.path.join(LOCAL_REPO_DIR, 'pose', 'ViTPose', 'checkpoints')
for p in [models_dst, reid_dst, pose_dst]:
    os.makedirs(p, exist_ok=True)

# Checkpoints on Drive — first match wins (very common: inside the repo clone on Drive):
#  - .../<project>/weights/{models,reid,pose}
#  - .../<project>/{models,reid,pose}
#  - .../<project>/<repo_name>/models  (same paths as git repo)
def _pick_dir(paths):
    for p in paths:
        if p and os.path.isdir(p):
            return p
    return None

_models_cands = [
    os.path.join(DRIVE_WEIGHTS_DIR, 'models'),
    os.path.join(DRIVE_PROJECT_DIR, 'models'),
    os.path.join(DRIVE_REPO_DIR, 'models'),
]
_reid_cands = [
    os.path.join(DRIVE_WEIGHTS_DIR, 'reid'),
    os.path.join(DRIVE_PROJECT_DIR, 'reid'),
    os.path.join(DRIVE_REPO_DIR, 'reid', 'centroids-reid', 'models'),
]
_pose_cands = [
    os.path.join(DRIVE_WEIGHTS_DIR, 'pose'),
    os.path.join(DRIVE_PROJECT_DIR, 'pose'),
    os.path.join(DRIVE_REPO_DIR, 'pose', 'ViTPose', 'checkpoints'),
]
base_models = _pick_dir(_models_cands)
base_reid = _pick_dir(_reid_cands)
base_pose = _pick_dir(_pose_cands)

if not base_models:
    raise FileNotFoundError(
        'Could not find a models/ folder on Drive. Put PARSeq + legibility files in ONE of:\n'
        + '\n'.join(f'  - {p}' for p in _models_cands)
    )

print('Using Drive models from:', base_models)
if base_reid:
    print('Using Drive reid from:', base_reid)
else:
    print('[WARN] ReID folder not found on Drive; tried:\n  ' + '\n  '.join(_reid_cands))
if base_pose:
    print('Using Drive pose from:', base_pose)
else:
    print('[WARN] Pose checkpoints folder not found; tried:\n  ' + '\n  '.join(_pose_cands))

src_map = [(base_models, models_dst)]
if base_reid:
    src_map.append((base_reid, reid_dst))
if base_pose:
    src_map.append((base_pose, pose_dst))
for src, dst in src_map:
    if os.path.isdir(src):
        run(f'rsync -a "{src}/" "{dst}/"')
    else:
        print(f'[WARN] Missing weights subfolder: {src}')

print('Weights staged.')


In [ ]:
# 4) Install dependencies (idempotent)
# Colab already includes CUDA runtime; install project deps in current runtime.
run('python -m pip install -q --upgrade pip', cwd=LOCAL_REPO_DIR)
run('python -m pip install -q -r requirements.txt', cwd=LOCAL_REPO_DIR)
run('python -m pip install -q gdown yacs pytorch-lightning', cwd=LOCAL_REPO_DIR)

# PARSeq deps (for str.py inference path)
run('python -m pip install -q -r str/parseq/requirements/inference.txt', cwd=LOCAL_REPO_DIR)
run('python -m pip install -q -e str/parseq', cwd=LOCAL_REPO_DIR)

# quick sanity
run('python - <<"PY"\nimport torch\nprint("torch", torch.__version__, "cuda", torch.cuda.is_available())\nPY', cwd=LOCAL_REPO_DIR)


In [ ]:
# 5) Ensure full-tracklet setting for final runs (optional patch)
cfg_path = os.path.join(LOCAL_REPO_DIR, 'configuration.py')
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg_text = f.read()

if not LIMIT_TRACKLETS:
    new_text = cfg_text.replace('soccer_net_max_tracklets = 5', 'soccer_net_max_tracklets = None')
    if new_text != cfg_text:
        with open(cfg_path, 'w', encoding='utf-8') as f:
            f.write(new_text)
        print('Updated soccer_net_max_tracklets to None for full run.')
    else:
        print('soccer_net_max_tracklets already not set to 5; no change.')
else:
    print('Keeping tracklet limit as configured.')


In [ ]:
# 6) Run pipeline with resume/force controls
# (Inlined subprocess capture so you see the full traceback even if an older `run()` is still in RAM.)
import subprocess as _sp
flags = []
if FORCE:
    flags.append('--force')
elif RESUME:
    flags.append('--resume')

cmd = f"python main.py SoccerNet {PART} {' '.join(flags)}".strip()
print('Running:', cmd)
p = _sp.run(
    cmd, shell=True, cwd=LOCAL_REPO_DIR, text=True,
    stdout=_sp.PIPE, stderr=_sp.STDOUT,
)
if p.stdout:
    print(p.stdout, end='')
if p.returncode != 0:
    raise RuntimeError(f'Command failed ({p.returncode}): {cmd}')


In [ ]:
# 7) Persist outputs + optionally notebook-side code changes back to Drive
local_out = os.path.join(LOCAL_REPO_DIR, 'out')
drive_out = os.path.join(DRIVE_PROJECT_DIR, 'out')
os.makedirs(drive_out, exist_ok=True)
if os.path.isdir(local_out):
    run(f'rsync -a "{local_out}/" "{drive_out}/"')
    print('Outputs synced to Drive:', drive_out)
else:
    print('No out/ folder found.')

# Optional: sync modified repo files back to persistent Drive clone (without deleting)
run(f'rsync -a "{LOCAL_REPO_DIR}/" "{DRIVE_REPO_DIR}/"')
print('Repo synced back to Drive clone.')
